# Explore anomaly scores

In [1]:
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sdss.metadata import MetaData
from anomaly.utils import specobjid_to_idx

meta = MetaData()

# Constants

In [2]:
mse_cols = ['mse', 'mse_filter_250', 'mse_97', 'mse_filter_250_97']
mse_rel_cols = ['mse_rel', 'mse_filter_250_rel', 'mse_97_rel', 'mse_filter_250_97_rel']

# Custom functions

In [10]:
def scores_arr_to_df(score_paths, bin_idx_to_specobjid):

    score_names = [score.split("/")[-1][:-4] for score in score_paths]
    objids_bin = bin_idx_to_specobjid[:, 1]

    scores_dict = {}

    for score_name, score_path in zip(score_names, score_paths):

        scores_dict[score_name] = np.load(score_path)[:, 0]

    scores_df = pd.DataFrame(data=scores_dict, index=objids_bin)
    scores_df.index.name = 'specobjid'
    scores_df.sort_values(by='mse', ascending=False, inplace=True)

    return scores_df[mse_cols + mse_rel_cols]


# Directories and Data

In [4]:
thesis_dir = "/home/elom/phd" 
data_dir = f"{thesis_dir}/code_phd"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
bin_ids = [f"bin_{i:02d}" for i in range(4)]
#
ch_4_dir = f"{thesis_dir}/chapters/04_figures"

In [5]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

meta_data_df = pd.read_csv(
    f"{spectra_dir}/drop_0_01_z_0_5_4_0_snr_inf.csv.gz",
    index_col="specobjid",
)

# Build csv with scores for each bin

```python
for bin_id in bin_ids:

    score_paths = glob.glob(f"{scores_dir}/{bin_id}/*.npy")

    bin_idx_to_specobjid = np.load(
        f"{spectra_dir}/"
        f"{bin_id}/{bin_id}_index_specobjid.npy"
    )

    scores_df = scores_arr_to_df(score_paths, bin_idx_to_specobjid)
    scores_df.to_csv(
        f"{scores_dir}/{bin_id}/scores_{bin_id}.csv.gz",
        index=True
    )
```

# Bind 03